# QIPS Scheduler — Analysis & Benchmarking
Reproduces and extends the results from Table 1 of the paper.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from qpso_engine import QPSOScheduler, Job, Node, compute_all_metrics
from qpso_engine.qpso import run_all_schedulers, fifo_schedule, fair_schedule, capacity_schedule

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Generate a synthetic workload

In [ ]:
def make_workload(n_jobs=8, n_nodes=3, seed=42):
    rng = random.Random(seed)
    nodes = [Node(id=f'node-{i+1}', total_cpu=8, total_mem=8192,
                  available_cpu=8, available_mem=8192, data_blocks=[])
             for i in range(n_nodes)]
    jobs = []
    names = ['WordCount','PageRank','TeraSort','HiveQuery','SparkML',
             'HBase','PigLatin','ETL','Flume','Sqoop']
    for i in range(n_jobs):
        ln = rng.randint(0, n_nodes-1)
        rt = rng.randint(20, 180)
        j = Job(id=f'J-{i+1:02d}', name=names[i % len(names)],
                priority=rng.choice([1.0, 0.75, 0.4, 0.15]),
                estimated_runtime=float(rt),
                cpu_demand=rng.randint(1,6),
                mem_demand=rng.choice([512,1024,2048,4096]),
                deadline_slack=float(rng.randint(int(rt*1.1), rt*4)),
                input_size_gb=round(rng.uniform(0.5,8.0),1),
                local_node_id=nodes[ln].id)
        jobs.append(j)
        nodes[ln].data_blocks.append(j.id)
    return jobs, nodes

jobs, nodes = make_workload(n_jobs=8, seed=42)
print(f'Generated {len(jobs)} jobs on {len(nodes)} nodes')
pd.DataFrame([j.model_dump() for j in jobs])[
    ['id','name','priority','estimated_runtime','cpu_demand','deadline_slack']]

## 2. Run all schedulers and compare

In [ ]:
weights = [1.0, 1.0, 1.5, 1.0, 1.2, 1.3]
results = run_all_schedulers(jobs, nodes, weights,
                              qpso_params={'n_particles':30,'max_iter':150,'beta':0.75,'seed':0})

rows = []
for name, r in results.items():
    m = r['metrics']
    rows.append({'Scheduler': name,
                 'Makespan (s)': m['makespan'],
                 'Avg Latency (s)': m['avg_latency'],
                 'Deadline Penalty': m['deadline_penalty'],
                 'Resource Usage (%)': m['resource_usage_pct'],
                 'Data Locality (%)': m['data_locality_pct'],
                 'Priority Satisfaction': m['priority_satisfaction']})

df = pd.DataFrame(rows).set_index('Scheduler')
print('\nPerformance comparison:')
df

## 3. QPSO convergence curve

In [ ]:
history = results['QIPS']['fitness_history']

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(history, color='#534AB7', linewidth=1.5)
ax.fill_between(range(len(history)), history, alpha=0.1, color='#534AB7')
ax.set_xlabel('Iteration')
ax.set_ylabel('Global best fitness')
ax.set_title('QPSO convergence')
plt.tight_layout()
plt.savefig('convergence.png', dpi=150)
plt.show()

## 4. Bar chart — metrics comparison (reproduces Fig. 2 from paper)

In [ ]:
metrics_to_plot = ['Makespan (s)', 'Avg Latency (s)', 'Deadline Penalty',
                   'Resource Usage (%)', 'Data Locality (%)', 'Priority Satisfaction']
colors = {'FIFO':'#B4B2A9','Fair':'#85B7EB','Capacity':'#5DCAA5','QIPS':'#534AB7'}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, metric in zip(axes.flat, metrics_to_plot):
    vals = [df.loc[s, metric] for s in df.index if s in colors]
    scheds = [s for s in df.index if s in colors]
    bars = ax.bar(scheds, vals, color=[colors[s] for s in scheds], width=0.5)
    ax.bar_label(bars, fmt='%.1f', padding=2, fontsize=9)
    ax.set_title(metric, fontsize=11)
    ax.set_ylim(0, max(vals)*1.25)

plt.suptitle('QIPS vs baseline schedulers', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Beta sensitivity analysis — how β affects convergence speed

In [ ]:
beta_values = [0.3, 0.5, 0.75, 1.0, 1.2]
fig, ax = plt.subplots(figsize=(9, 4))

for beta in beta_values:
    sched = QPSOScheduler(n_particles=20, max_iter=100, beta=beta, seed=7)
    _, fit, hist = sched.optimize(jobs, nodes, weights)
    ax.plot(hist, label=f'β={beta}  (final={fit:.2f})', linewidth=1.4)

ax.set_xlabel('Iteration')
ax.set_ylabel('Global best fitness')
ax.set_title('Effect of β on convergence')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('beta_sensitivity.png', dpi=150)
plt.show()

## 6. Swarm size sensitivity

In [ ]:
swarm_sizes = [5, 10, 20, 30, 50]
fig, ax = plt.subplots(figsize=(9, 4))

for n_p in swarm_sizes:
    sched = QPSOScheduler(n_particles=n_p, max_iter=100, beta=0.75, seed=3)
    _, fit, hist = sched.optimize(jobs, nodes, weights)
    ax.plot(hist, label=f'n={n_p}  (final={fit:.2f})', linewidth=1.4)

ax.set_xlabel('Iteration')
ax.set_ylabel('Global best fitness')
ax.set_title('Effect of swarm size on convergence')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('swarm_sensitivity.png', dpi=150)
plt.show()

## 7. Multi-run benchmark (statistical summary)

In [ ]:
N_RUNS = 15
all_rows = []

for run in range(N_RUNS):
    j, n = make_workload(n_jobs=8, seed=run*17)
    r = run_all_schedulers(j, n, weights,
                            qpso_params={'n_particles':25,'max_iter':80,'beta':0.75,'seed':run})
    for name, data in r.items():
        m = data['metrics']
        all_rows.append({'run': run, 'scheduler': name,
                         'makespan': m['makespan'],
                         'latency': m['avg_latency'],
                         'locality': m['data_locality_pct'],
                         'priority': float(m['priority_satisfaction'])})

big = pd.DataFrame(all_rows)
summary = big.groupby('scheduler').agg(['mean','std']).round(2)
print('\nStatistical summary across', N_RUNS, 'runs:')
summary